# BTIS3043 Final Assessment — Intelligent eBook Query System

This notebook provides the executable evidence for the two fixed assessment scenarios. The implementation follows the required workflow:

**Scenario request → dataset-specific predicate query → predicate-only results → fuzzy evaluation → fuzzy-enhanced ranking → comparison and analysis**

The three datasets are processed **separately rather than merged** because they contain different fields and different types of evidence. The notebook therefore preserves dataset-specific search fields while applying a common reasoning process: crisp predicates first determine whether a record is a valid candidate, then fuzzy reasoning evaluates the degree of suitability of the accepted candidates.

Concise interpretation is included after the main outputs so that the effect of dataset size, metadata structure, missing evidence, predicate design and fuzzy ranking can be seen directly from the executable results.

In [1]:
import pandas as pd
from IPython.display import display

from src.data_loader import load_datasets, dataset_profile, DATASET_SPECS
from src.knowledge_base import get_scenario
from src.predicate_engine import basic_predicate_query, combined_predicate_query
from src.fuzzy_engine import apply_fuzzy_evaluation, explain_record
from src.evaluation import (
    run_scenario,
    comparison_summary,
    top_results,
    sensitivity_summary,
)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

## 1. Dataset and knowledge representation

Each catalogue keeps its own structure instead of being forced into one common schema.

- **Dataset A — Existing eBook Collection:** a very small current collection. Search evidence is mainly the title, with copyright year and unit net price available for fuzzy evaluation.
- **Dataset B — Academic eBook Catalogue:** the largest catalogue. It provides the title plus four discipline levels and an eBook-format field, but no comparable price field.
- **Dataset C — eBook Acquisition Catalogue:** a medium-sized acquisition catalogue. It provides title, category, discipline, eBook format and licensing price evidence.

This representation matters because a larger catalogue or richer discipline metadata can create more predicate matches, while format and price fields allow more fuzzy preferences to be evaluated. The following profile is generated directly from the loaded data.

In [2]:
datasets = load_datasets(data_dir="data")
profile = dataset_profile(datasets)
display(profile)

,Dataset,Role,Records,Search fields,Discipline detail,Year evidence,Format evidence,Comparable price,Selected price field
0,A,Current/existing collection,9,Title,None; title only,Yes,No,Yes,Unit Net Price
1,B,Academic/vendor catalogue,1743,"Title, Discipline (Level 1), Discipline (Level 2), Discipline (Level 3), Discipline (L...",Four discipline levels,Yes,Yes,No,Not available
2,C,Potential acquisition catalogue,807,"Title, Category, Discipline",Category + discipline,Yes,Yes,Yes,Single user / 1-Year


## 2. Predicate design: basic and combined predicates

The system demonstrates both **basic** and **combined** predicates.

- **Basic predicate:** accepts only records that match the direct main topic.
- **Combined predicate:** broadens the candidate set by combining direct and supporting relationships with Boolean `OR`.

For Scenario 1:

`Direct_AI OR Programming_Support OR Mathematical_Support`

For Scenario 2:

`Direct_Security OR Security_Related_Support`

The predicates are deliberately used for topic/category acceptance, while gradual preferences such as recency, format suitability and affordability are left to the fuzzy stage. This prevents a record from being rejected simply because it is slightly older or more expensive.

The generic word **security** receives an additional computing-context check. This reduces false positives such as *Food Security* while still accepting computing titles such as *Security in Computing*.

In [3]:
for sid in ("S1", "S2"):
    scenario = get_scenario(sid)
    print(f"{sid}: {scenario['name']}")
    print("Combined predicate expression:", scenario["predicate_expression"])
    print()

S1: Artificial Intelligence, Programming and Mathematical Foundations
Combined predicate expression: Direct_AI OR Programming_Support OR Mathematical_Support

S2: Cybersecurity and Secure Computing
Combined predicate expression: Direct_Security OR Security_Related_Support, with a computing-context guard for the generic word 'security'



In [4]:
predicate_demo_rows = []
for sid in ("S1", "S2"):
    for key in ("A", "B", "C"):
        basic = basic_predicate_query(datasets[key], key, sid)
        combined = combined_predicate_query(datasets[key], key, sid)
        predicate_demo_rows.append({
            "Scenario": sid,
            "Dataset": key,
            "Basic direct matches": len(basic),
            "Combined predicate matches": len(combined),
            "Additional candidates from combined logic": len(combined) - len(basic),
        })

display(pd.DataFrame(predicate_demo_rows))

,Scenario,Dataset,Basic direct matches,Combined predicate matches,Additional candidates from combined logic
0,S1,A,0,0,0
1,S1,B,6,234,228
2,S1,C,5,110,105
3,S2,A,1,1,0
4,S2,B,9,9,0
5,S2,C,5,7,2


### Security false-positive check

A generic word such as *security* is not enough by itself. The following small test shows the contextual guard: *Security in Computing* is accepted, while *Food Security* is rejected.

In [5]:
security_test = pd.DataFrame({
    "Title": ["Understanding Food Security", "Security in Computing"],
    "Copyright Year": [2024, 2024],
    "Unit Net Price": [100.0, 100.0],
    "_source_order": [1, 2],
})
security_test_result = combined_predicate_query(security_test, "A", "S2")
display(security_test_result[["Title", "Relationship", "Matched_Terms"]])

,Title,Relationship,Matched_Terms
0,Security in Computing,Direct Security,"security in computing, security"


## 3. Fuzzy reasoning design

Predicate reasoning answers **which records satisfy the scenario conditions**. Fuzzy reasoning then answers **to what degree the accepted records are suitable**.

Four fuzzy preference components are used where evidence is available:

1. **Topic relevance** — direct relationships receive the strongest membership. Programming, mathematical or security-related support receives a lower but still meaningful membership. Title evidence is treated as stronger than metadata-only evidence.
2. **Recency** — a piecewise-linear membership function gradually decreases as publication age increases, using 2026 as the assessment year.
3. **Format suitability** — common eBook formats such as ePub/PDF receive the strongest membership. Adobe Reader remains a suitable digital format rather than being treated as unknown.
4. **Affordability** — a catalogue-relative membership based on the 25th and 90th percentiles of the selected comparable price field.

The default aggregation weights are:

- relevance = `0.45`
- recency = `0.25`
- format suitability = `0.15`
- affordability = `0.15`

Relevance receives the largest weight because topic fit is the primary purpose of the search. Recency is the next priority, while format and affordability refine the ranking when the evidence exists.

If a component is unavailable, it is **not replaced by an artificial neutral score**. Instead, the missing component is excluded and the remaining weights are re-normalised:

\[
Suitability = \frac{\sum_{i\in A} w_i\mu_i}{\sum_{i\in A} w_i}
\]

where \(A\) is the set of fuzzy components for which evidence is available. This is important because Dataset A lacks a useful format field, Dataset B lacks comparable price evidence, and Dataset C provides both.

In [6]:
for sid in ("S1", "S2"):
    print(sid, get_scenario(sid)["weights"])

S1 {'relevance': 0.45, 'recency': 0.25, 'format': 0.15, 'affordability': 0.15}
S2 {'relevance': 0.45, 'recency': 0.25, 'format': 0.15, 'affordability': 0.15}


# 4. Fixed Scenario 1 — Artificial Intelligence, Programming and Mathematical Foundations

Scenario 1 searches for three justified relationships: **Direct AI**, **Programming Support** and **Mathematical Support**. The combined predicate creates the final candidate set, and fuzzy reasoning then ranks those candidates using the available relevance, recency, format and affordability evidence.

The output displays up to five fuzzy-ranked records per dataset, as required. If a dataset returns fewer than five records, the notebook explains why.

In [7]:
s1 = run_scenario(datasets, "S1")
s1_summary = comparison_summary(datasets, s1, "S1")
display(s1_summary)

,Scenario,Dataset,Dataset size,Basic direct matches,Combined predicate matches,Match rate %,Top fuzzy score,Top relationship,Search structure,Format evidence,Price evidence,Fuzzy ranking changed order
0,S1,A,9,0,0,0.00,NaN,No match,None; title only,No,Yes,False
1,S1,B,1743,6,234,13.43,1.00,Direct AI,Four discipline levels,Yes,No,True
2,S1,C,807,5,110,13.63,0.95,Direct AI,Category + discipline,Yes,Yes,True


In [8]:
for key in ("A", "B", "C"):
    print()
    print(f"Dataset {key}: {DATASET_SPECS[key]['name']}")
    print(f"Predicate-only matches: {len(s1[key]['predicate'])}")
    if s1[key]["predicate"].empty:
        print("No record satisfied the Scenario 1 predicate in this dataset.")
    else:
        display(top_results(s1[key]["fuzzy"], 5))


Dataset A: Existing eBook Collection
Predicate-only matches: 0
No record satisfied the Scenario 1 predicate in this dataset.

Dataset B: Academic eBook Catalogue
Predicate-only matches: 234


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,1,0,Artificial Intelligence: A Guide to Intelligent Systems,Direct AI,"artificial intelligence, intelligent systems","Title, Discipline (Level 3), Discipline (Level 4)",1.00,1.0,1.0,NaN,1.0000,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
1,2,3,1,"Artificial Intelligence: A Modern Approach, Global Edition\n",Direct AI,artificial intelligence,"Title, Discipline (Level 3), Discipline (Level 4)",1.00,0.8,1.0,NaN,0.9412,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
2,3,35,32,"C++ How to Program, Global Edition",Programming Support,"c++, programming","Title, Discipline (Level 3), Discipline (Level 4)",0.81,1.0,1.0,NaN,0.8994,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
3,4,36,32,"C++ How to Program, Global Edition",Programming Support,"c++, programming","Title, Discipline (Level 3), Discipline (Level 4)",0.81,1.0,1.0,NaN,0.8994,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
4,5,128,123,"Introduction to Java Programming and Data Structures, Global Edition",Programming Support,"java, programming, data structures","Title, Discipline (Level 3), Discipline (Level 4)",0.81,1.0,1.0,NaN,0.8994,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...



Dataset C: eBook Acquisition Catalogue
Predicate-only matches: 110


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,2,1,"Artificial Intelligence, 2e",Direct AI,artificial intelligence,Title,1.00,0.8,1.0,1.0000,0.9500,Main positive driver: topic relevance.
1,2,3,1,China’s Robots,Direct AI,robots,Title,1.00,0.8,1.0,1.0000,0.9500,Main positive driver: topic relevance.
2,3,5,2,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",Direct AI,"artificial intelligence, programming","Title, Discipline",1.00,0.7,1.0,1.0000,0.9250,Main positive driver: topic relevance.
3,4,4,0,Introduction to Artificial Intelligence: A Business Perspective,Direct AI,artificial intelligence,Title,1.00,1.0,1.0,0.4973,0.9246,Main positive driver: topic relevance.
4,5,24,19,Android Boot Camp for Developers Using Java®,Programming Support,"java, programming","Title, Discipline",0.81,1.0,1.0,0.9227,0.9029,Main positive driver: topic relevance.


### Scenario 1 result interpretation

The Scenario 1 outputs show a strong effect from both **catalogue size** and **metadata richness**.

**Dataset A** contains only 9 records and returns **0 matches**. Because its searchable evidence is mainly the title, it cannot benefit from discipline/category metadata when an AI, programming or mathematical relationship is not explicitly expressed in the title. The zero result is therefore meaningful evidence of limited Scenario 1 coverage in the existing collection rather than a system failure.

**Dataset B** contains 1,743 records and returns **234 combined-predicate matches (13.43%)**, compared with only 6 basic direct-AI matches. The large increase comes from programming and mathematical support plus the four discipline levels. Fuzzy reasoning is useful because the candidate set is large: for example, recent C++ and Java programming titles move far above their predicate-only positions because they combine useful support relevance with strong recency and format evidence. The top result, *Artificial Intelligence: A Guide to Intelligent Systems*, receives a perfect fuzzy score from direct AI relevance, very recent publication evidence and strong format suitability; affordability is unavailable and is therefore excluded.

**Dataset C** contains 807 records and returns **110 combined-predicate matches (13.63%)**. Its category/discipline fields support broader retrieval, while the `Single user / 1-Year` price provides an additional fuzzy discriminator. *Artificial Intelligence, 2e* becomes the top result with a fuzzy score of **0.9500** because it combines direct AI relevance, suitable format and strong affordability despite not receiving the maximum recency membership. This illustrates how fuzzy reasoning balances several gradual preferences instead of treating one attribute as an absolute requirement.

### Scenario 1 selected decision explanations

The following explanations show why a top-ranked record passed the crisp predicate and which fuzzy memberships contributed to the final score. This makes the ranking traceable rather than reporting only a final number.

In [9]:
for key in ("B", "C"):
    top = s1[key]["fuzzy"].iloc[0]
    print(f"Dataset {key} top-ranked record:")
    print(top["Title"])
    print(explain_record(top, key))
    print()

Dataset B top-ranked record:
Artificial Intelligence: A Guide to Intelligent Systems
Relationship=Direct AI; matched terms=artificial intelligence, intelligent systems; matched fields=Title, Discipline (Level 3), Discipline (Level 4); relevance=1.00; recency=1.00; format=1.00; affordability=NA; final=1.0000; Main positive driver: topic relevance. Unavailable evidence excluded and weights re-normalised: affordability.

Dataset C top-ranked record:
Artificial Intelligence, 2e
Relationship=Direct AI; matched terms=artificial intelligence; matched fields=Title; relevance=1.00; recency=0.80; format=1.00; affordability=1.00; final=0.9500; price field=Single user / 1-Year; Main positive driver: topic relevance.



# 5. Fixed Scenario 2 — Cybersecurity and Secure Computing

Scenario 2 distinguishes **Direct Security** from **Security-Related Support**. Explicit cybersecurity, computer-security and network-security concepts receive the strongest topic relationship, while related areas such as cryptography, forensics and incident response remain relevant but do not automatically receive the same relevance membership.

Dataset A represents the existing/current collection, so all relevant matches are displayed. Up to ten records are shown for the larger catalogues. The smaller number of security matches also allows the effect of recency, format and affordability to be observed more clearly.

In [10]:
s2 = run_scenario(datasets, "S2")
s2_summary = comparison_summary(datasets, s2, "S2")
display(s2_summary)

,Scenario,Dataset,Dataset size,Basic direct matches,Combined predicate matches,Match rate %,Top fuzzy score,Top relationship,Search structure,Format evidence,Price evidence,Fuzzy ranking changed order
0,S2,A,9,1,1,11.11,0.8235,Direct Security,None; title only,No,Yes,False
1,S2,B,1743,9,9,0.52,1.0000,Direct Security,Four discipline levels,Yes,No,True
2,S2,C,807,5,7,0.87,1.0000,Direct Security,Category + discipline,Yes,Yes,True


In [11]:
for key in ("A", "B", "C"):
    print()
    print(f"Dataset {key}: {DATASET_SPECS[key]['name']}")
    print(f"Predicate-only matches: {len(s2[key]['predicate'])}")
    if key == "A":
        print("Dataset A represents the existing/current collection; all relevant current holdings are shown.")
    if s2[key]["predicate"].empty:
        print("No record satisfied the Scenario 2 predicate in this dataset.")
    else:
        display(top_results(s2[key]["fuzzy"], 10))


Dataset A: Existing eBook Collection
Predicate-only matches: 1
Dataset A represents the existing/current collection; all relevant current holdings are shown.


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,1,0,Security in Computing,Direct Security,"security in computing, security",Title,1.0,1.0,NaN,0.0,0.8235,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...



Dataset B: Academic eBook Catalogue
Predicate-only matches: 9


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,2,1,"Computer Security: Principles and Practice, Global Edition",Direct Security,"computer security, security",Title,1.0,1.00,1.00,NaN,1.0000,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
1,2,9,7,Security in Computing,Direct Security,"security in computing, security",Title,1.0,1.00,1.00,NaN,1.0000,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
2,3,4,1,"Cryptography and Network Security: Principles and Practice, Global Edition",Direct Security,"network security, security, computer security, cryptography","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.80,0.85,NaN,0.9147,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
3,4,7,3,"Network Security Essentials: Applications and Standards, Global Edition",Direct Security,"network security, security, computer security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.54,0.85,NaN,0.8382,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
4,5,3,-2,"Computer Security: Principles and Practice, Global Edition",Direct Security,"computer security, security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.46,0.85,NaN,0.8147,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
5,6,1,-5,"Boyle: Corporate Computer Security, Global Edition",Direct Security,"computer security, security, network security","Title, Discipline (Level 4)",1.0,0.26,0.85,NaN,0.7559,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
6,7,6,-1,"Business Data Networks and Security, Global Edition",Direct Security,security,Title,1.0,0.26,0.85,NaN,0.7559,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
7,8,5,-3,Introduction to Computer Security,Direct Security,"computer security, security",Title,1.0,0.22,0.85,NaN,0.7441,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
8,9,8,-1,Practical Cryptology and Web Security,Direct Security,security,Title,1.0,0.10,0.85,NaN,0.7088,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...



Dataset C: eBook Acquisition Catalogue
Predicate-only matches: 7


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,5,4,Security Awareness: Applying Practical Cybersecurity in Your World,Direct Security,"cybersecurity, security",Title,1.00,1.0,1.0,1.0000,1.0000,Main positive driver: topic relevance.
1,2,3,1,Management of Cybersecurity,Direct Security,"cybersecurity, security",Title,1.00,1.0,1.0,0.7183,0.9577,Main positive driver: topic relevance.
2,3,1,-2,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),Direct Security,"cybersecurity, security",Title,1.00,1.0,1.0,0.3536,0.9030,Main positive driver: topic relevance.
3,4,2,-2,CompTIA Security+ Guide to Network Security Fundamentals,Direct Security,"network security, security",Title,1.00,1.0,1.0,0.3536,0.9030,Main positive driver: topic relevance.
4,5,4,-1,Principles of Information Security,Direct Security,"information security, security",Title,1.00,0.8,1.0,0.3536,0.8530,Main positive driver: topic relevance.
5,6,6,0,Guide to Computer Forensics and Investigations,Security-Related Support,computer forensics,Title,0.78,1.0,1.0,0.3536,0.8040,Main positive driver: topic relevance.
6,7,7,0,Principles of Incident Response & Disaster Recovery,Security-Related Support,incident response,Title,0.78,0.8,1.0,0.3536,0.7540,Main positive driver: topic relevance.


### Scenario 2 result interpretation

Scenario 2 returns far fewer candidates than Scenario 1, which indicates that **cybersecurity coverage is more limited** in these catalogues.

**Dataset A** returns only **1 relevant current-collection record** from 9 records: *Security in Computing*. Its relevance and recency memberships are both high, but the title has no configured format evidence and its affordability membership is low relative to Dataset A. Its final fuzzy score of **0.8235** therefore demonstrates a real trade-off: a highly relevant, current title can still lose suitability points because another preference is weak.

**Dataset B** returns **9 matches from 1,743 records (0.52%)**. Although all nine satisfy the security predicate, fuzzy ordering changes their positions because recency and format differ. *Security in Computing* moves from predicate rank 9 to fuzzy rank 2, showing that fuzzy reasoning improves ordering even when the crisp relationship class is the same. Dataset B has no comparable price field, so affordability is excluded rather than estimated.

**Dataset C** returns **7 matches from 807 records (0.87%)**, including direct cybersecurity titles and related secure-computing support. *Security Awareness: Applying Practical Cybersecurity in Your World* ranks first with **1.0000** because all available fuzzy components are strong. In contrast, *Guide to Computer Forensics and Investigations* is retained as **Security-Related Support**, demonstrating that the system can recognise an important related area without giving it the same base relevance as an explicit cybersecurity title.

### Selected affordability evidence

Affordability is used only when comparable price evidence exists.

- **Dataset A:** uses `Unit Net Price`.
- **Dataset B:** has no comparable price field, so affordability is excluded and the remaining fuzzy weights are re-normalised.
- **Dataset C:** uses `Single user / 1-Year` as the selected licence arrangement.

The affordability thresholds are catalogue-relative rather than a claim that a particular price is universally cheap or expensive.

In [12]:
for sid, outputs in [("S1", s1), ("S2", s2)]:
    print(sid)
    for key in ("A", "B", "C"):
        field = DATASET_SPECS[key]["price_field"]
        thresholds = outputs[key]["affordability_thresholds"]
        print(f"  Dataset {key}: price field={field or 'Not available'}, thresholds={thresholds}")

S1
  Dataset A: price field=Unit Net Price, thresholds=None
  Dataset B: price field=Not available, thresholds=None
  Dataset C: price field=Single user / 1-Year, thresholds={'low': 86.688, 'high': 196.4445}
S2
  Dataset A: price field=Unit Net Price, thresholds={'low': 212.86, 'high': 638.648}
  Dataset B: price field=Not available, thresholds=None
  Dataset C: price field=Single user / 1-Year, thresholds={'low': 86.688, 'high': 196.4445}


### Scenario 2 selected decision explanations

The following explanations show the predicate evidence, relationship class, fuzzy component scores and the main positive driver for the highest-ranked security record in each dataset. They also make missing evidence explicit where format or affordability cannot be evaluated.

In [13]:
for key in ("A", "B", "C"):
    if not s2[key]["fuzzy"].empty:
        top = s2[key]["fuzzy"].iloc[0]
        print(f"Dataset {key} top-ranked record:")
        print(top["Title"])
        print(explain_record(top, key))
        print()

Dataset A top-ranked record:
Security in Computing
Relationship=Direct Security; matched terms=security in computing, security; matched fields=Title; relevance=1.00; recency=1.00; format=NA; affordability=0.00; final=0.8235; price field=Unit Net Price; Main positive driver: topic relevance. Unavailable evidence excluded and weights re-normalised: format suitability.

Dataset B top-ranked record:
Computer Security: Principles and Practice, Global Edition
Relationship=Direct Security; matched terms=computer security, security; matched fields=Title; relevance=1.00; recency=1.00; format=1.00; affordability=NA; final=1.0000; Main positive driver: topic relevance. Unavailable evidence excluded and weights re-normalised: affordability.

Dataset C top-ranked record:
Security Awareness: Applying Practical Cybersecurity in Your World
Relationship=Direct Security; matched terms=cybersecurity, security; matched fields=Title; relevance=1.00; recency=1.00; format=1.00; affordability=1.00; final=1.00

## 6. Predicate-only vs fuzzy-enhanced comparison

`Predicate_Rank` orders accepted records using crisp relationship priority and stable catalogue order. `Fuzzy_Rank` reorders those same candidates using gradual suitability.

`Rank_Change > 0` means a record moved upward after fuzzy evaluation, while a negative value means it moved downward. A change therefore shows that fuzzy reasoning adds information beyond simple predicate acceptance. This comparison is especially useful when many records satisfy the same crisp relationship but differ in recency, format or affordability.

In [14]:
rank_change_examples = []
for sid, outputs in [("S1", s1), ("S2", s2)]:
    for key in ("A", "B", "C"):
        f = outputs[key]["fuzzy"]
        if not f.empty:
            moved = f.loc[f["Rank_Change"] != 0, [
                "Title", "Predicate_Rank", "Fuzzy_Rank", "Rank_Change", "Fuzzy_Score"
            ]].head(3).copy()
            moved.insert(0, "Dataset", key)
            moved.insert(0, "Scenario", sid)
            rank_change_examples.append(moved)

if rank_change_examples:
    display(pd.concat(rank_change_examples, ignore_index=True))
else:
    print("No rank changes occurred.")

,Scenario,Dataset,Title,Predicate_Rank,Fuzzy_Rank,Rank_Change,Fuzzy_Score
0,S1,B,"Artificial Intelligence: A Modern Approach, Global Edition\n",3,2,1,0.9412
1,S1,B,"C++ How to Program, Global Edition",35,3,32,0.8994
2,S1,B,"C++ How to Program, Global Edition",36,4,32,0.8994
3,S1,C,"Artificial Intelligence, 2e",2,1,1,0.9500
4,S1,C,China’s Robots,3,2,1,0.9500
5,S1,C,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",5,3,2,0.9250
6,S2,B,"Computer Security: Principles and Practice, Global Edition",2,1,1,1.0000
7,S2,B,Security in Computing,9,2,7,1.0000
8,S2,B,"Cryptography and Network Security: Principles and Practice, Global Edition",4,3,1,0.9147
9,S2,C,Security Awareness: Applying Practical Cybersecurity in Your World,5,1,4,1.0000


### Interpretation of ranking changes

The displayed rank changes confirm that fuzzy reasoning is doing more than attaching a score to an unchanged list. In Scenario 1, some programming-support records move substantially upward because strong recency and format suitability compensate for their lower topic-relevance membership compared with direct AI titles. In Scenario 2, *Security in Computing* in Dataset B moves from predicate rank 9 to fuzzy rank 2 because its recent publication evidence and strong format suitability make it more useful than several older security titles.

The opposite movement is also meaningful. A record can move downward when it satisfies the same crisp topic relationship as other records but has weaker gradual preferences. Therefore, predicate reasoning remains useful for transparent acceptance, while fuzzy reasoning improves the practical order of the accepted shortlist.

## 7. Cross-dataset evaluation

The combined summary compares both scenarios across all three catalogues. It reports dataset size, basic and combined predicate matches, match rate, top fuzzy score, discipline structure, format evidence, price evidence and whether fuzzy reasoning changed the ordering.

The comparison is used to evaluate how **dataset size, topic coverage, available attributes, discipline detail, price availability and number of matches** affect the outcome. Importantly, fuzzy scores are interpreted mainly **within each dataset** because the datasets do not always provide the same evidence. For example, Dataset B can rank records without affordability, while Dataset C can use both format and licensing price.

In [15]:
cross_dataset = pd.concat([s1_summary, s2_summary], ignore_index=True)
display(cross_dataset)

,Scenario,Dataset,Dataset size,Basic direct matches,Combined predicate matches,Match rate %,Top fuzzy score,Top relationship,Search structure,Format evidence,Price evidence,Fuzzy ranking changed order
0,S1,A,9,0,0,0.00,NaN,No match,None; title only,No,Yes,False
1,S1,B,1743,6,234,13.43,1.0000,Direct AI,Four discipline levels,Yes,No,True
2,S1,C,807,5,110,13.63,0.9500,Direct AI,Category + discipline,Yes,Yes,True
3,S2,A,9,1,1,11.11,0.8235,Direct Security,None; title only,No,Yes,False
4,S2,B,1743,9,9,0.52,1.0000,Direct Security,Four discipline levels,Yes,No,True
5,S2,C,807,5,7,0.87,1.0000,Direct Security,Category + discipline,Yes,Yes,True


### Cross-dataset interpretation

Three patterns are visible in the combined summary.

1. **Topic coverage matters as much as dataset size.** Dataset B is the largest catalogue and Scenario 1 produces 234 matches, while Scenario 2 produces only 9. Dataset C shows the same pattern: 110 Scenario 1 matches but only 7 Scenario 2 matches. The difference therefore reflects subject coverage, not only total record count.

2. **Richer metadata increases retrieval opportunities.** Dataset A has only title-based search evidence and returns no Scenario 1 candidates. Datasets B and C can also use discipline/category fields, so books can be found even when the scenario relationship is represented in metadata rather than explicitly in the title.

3. **Available fuzzy evidence changes how suitability is calculated.** Dataset A has price but no useful format field; Dataset B has format but no comparable price; Dataset C has both. Missing components are re-normalised, so a fuzzy score such as `1.0000` should not be treated as an identical absolute measurement across all three datasets. The safest interpretation is to compare rankings primarily within the same catalogue and then use the cross-dataset table to explain why the evidence differs.

## 8. Small sensitivity check

The fuzzy weights are a justified design choice, not an objective truth. To test whether the main ranking is overly dependent on one weight set, Dataset C is re-ranked with a more relevance-heavy alternative:

`relevance=0.60, recency=0.20, format=0.10, affordability=0.10`

If the leading results remain stable, the ranking is reasonably robust to this change. If records swap positions, the change identifies cases where the final order depends more strongly on preference priorities.

In [16]:
alternative_weights = {
    "relevance": 0.60,
    "recency": 0.20,
    "format": 0.10,
    "affordability": 0.10,
}

print("Scenario 1, Dataset C")
display(sensitivity_summary(s1["C"]["fuzzy"], alternative_weights, top_n=5))

print("Scenario 2, Dataset C")
display(sensitivity_summary(s2["C"]["fuzzy"], alternative_weights, top_n=7))

Scenario 1, Dataset C


,_source_order,Title,Fuzzy_Rank,Fuzzy_Score,Alternative_Rank,Alternative_Score,Sensitivity_Rank_Change
21,164,"Artificial Intelligence, 2e",1,0.9500,1,0.96000,0
23,208,China’s Robots,2,0.9500,2,0.96000,0
101,778,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",3,0.9250,4,0.94000,-1
56,422,Introduction to Artificial Intelligence: A Business Perspective,4,0.9246,3,0.94973,1
19,156,Android Boot Camp for Developers Using Java®,5,0.9029,5,0.87827,0


Scenario 2, Dataset C


,_source_order,Title,Fuzzy_Rank,Fuzzy_Score,Alternative_Rank,Alternative_Score,Sensitivity_Rank_Change
6,678,Security Awareness: Applying Practical Cybersecurity in Your World,1,1.0000,1,1.00000,0
3,442,Management of Cybersecurity,2,0.9577,2,0.97183,0
0,229,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),3,0.9030,3,0.93536,0
1,232,CompTIA Security+ Guide to Network Security Fundamentals,4,0.9030,4,0.93536,0
5,668,Principles of Information Security,5,0.8530,5,0.89536,0
2,331,Guide to Computer Forensics and Investigations,6,0.8040,6,0.80336,0
4,667,Principles of Incident Response & Disaster Recovery,7,0.7540,7,0.76336,0


### Sensitivity interpretation

The sensitivity check suggests that the main conclusions are reasonably stable. For Scenario 1 Dataset C, the top two records remain unchanged under the relevance-heavy weights, while the third and fourth positions swap. This indicates that the leading recommendations are robust, but some middle positions depend on the chosen preference priorities.

For Scenario 2 Dataset C, the displayed top seven positions remain unchanged under the alternative weighting. This provides additional evidence that the security ranking is not being produced only by one arbitrary weight configuration. The check does not prove that the weights are objectively correct, but it makes the design choice more transparent and identifies where expert judgement could matter.

## 9. Implementation strengths, limitations and improvement directions

**Strengths**

- Dataset-specific search fields preserve the evidence available in each catalogue.
- Basic and combined predicates are both demonstrated explicitly.
- Direct and supporting relationships are represented separately.
- Fuzzy membership functions model gradual preferences instead of forcing arbitrary pass/fail thresholds.
- Missing evidence is excluded and the remaining weights are re-normalised.
- `Predicate_Rank`, `Fuzzy_Rank` and `Rank_Change` make the effect of fuzzy reasoning visible.
- The contextual security rule reduces an important false-positive case.
- Selected decision explanations make the system traceable and reproducible.

**Limitations**

- Keyword rules cannot represent every synonym, context or semantic relationship.
- The broad discipline metadata in larger catalogues can retrieve supporting books that are less directly related than title matches.
- Affordability is relative to each catalogue because no departmental budget is supplied.
- Fuzzy membership boundaries and weights still require expert judgement.
- Cross-dataset fuzzy scores are not perfectly equivalent because the available evidence differs.

**Possible improvements**

Future work could validate the keyword groups, membership boundaries and weights with DCS staff or librarians, add controlled vocabulary or ontology terms to reduce missed synonyms, use a real acquisition budget if one becomes available, and test semantic retrieval methods against the current transparent rule-based baseline. Machine learning is intentionally not required for this prototype.

## 10. Conclusion

The prototype demonstrates that crisp predicate reasoning and fuzzy reasoning serve different but complementary purposes. Predicate rules provide a transparent way to identify records that satisfy each scenario, while fuzzy memberships convert gradual preferences such as relevance, recency, format suitability and affordability into a more useful ranking.

The results also show that retrieval quality depends strongly on the available data. Larger catalogues and richer discipline/category metadata create more opportunities to identify relevant support records, while format and price fields allow more detailed fuzzy evaluation. The Scenario 2 security-context guard further shows that predicate precision remains important because fuzzy reasoning should rank relevant candidates rather than compensate for clearly irrelevant matches.

Overall, the combined predicate–fuzzy approach is more informative than predicate-only filtering because it preserves clear acceptance logic while producing an explainable ranked shortlist. Its main limitations are the dependence on curated keywords, catalogue-relative affordability and expert-selected fuzzy parameters, all of which provide clear directions for future improvement.